# Demo 1.4: GoalScope when the LM picks the next sub-task itself

**Insight.** In Demo 1 the next sub-task is written in the prompt. Here the prompt only asks for three *random but
different* computations on the input and does not say which. The first two are prefilled; the third is the LM's own
choice. If the goal heads encode the next sub-task, GoalScope should read out the computation the LM is about to
write, before it writes any of it.

**Setup.** For each datatype (`int`, `str`, `list[int]`, `list[str]`), 8 examples. The user turn asks for three
different computations on the input, stored in `a`, `b`, `c`. The assistant turn is prefilled with two computations
drawn at random from the evaluation corpus's task bank (`src/nrgs/data.py::QTASKS`), then `c =`, e.g.
```python
def func(words: list[str]):
    a = any(w == w[::-1] for w in words)
    b = sum(len(w) for w in words)
    c =
```
The last four prompt tokens (the newline after `b`, the indentation, `c` and `=`) are the pivots. The LM then writes
the `c` line on its own; that line is what the judge compares every readout against.

**Readout (GoalScope).** As in Demo 1: at each pivot, the 122 goal heads' summed write steers the neutral probe
*"What are you doing at this line of the code?"* (faithfulness) and the empty C++ stub `auto func (auto value){`
(causality, the causality_auto probe). An LM judge (Claude) checks whether the description describes, or the C++ body
implements, the computation in the LM's `c` line.

Needs one 80 GB GPU (Qwen2.5-Coder-32B-Instruct in bf16). The executed outputs are saved in this file.

In [1]:
# repo root = first parent dir that contains src/
import os, sys
_d = os.getcwd()
while _d != '/' and not os.path.isdir(os.path.join(_d, 'src')):
    _d = os.path.dirname(_d)
sys.path.insert(0, _d); os.chdir(_d)
print('repo root:', _d)

# no progress bars: keeps tqdm / HF widget outputs out of the saved notebook
os.environ['TQDM_DISABLE'] = '1'
import functools, tqdm.std, tqdm.notebook, transformers, huggingface_hub
for _c in (tqdm.std.tqdm, tqdm.notebook.tqdm):   # tqdm.notebook ignores TQDM_DISABLE, so force disable=True
    _c.__init__ = functools.partialmethod(_c.__init__, disable=True)
transformers.utils.logging.disable_progress_bar()
transformers.utils.logging.set_verbosity_error()      # no generation-config warnings
huggingface_hub.utils.disable_progress_bars()

# colors as in Demo 4: the LM's own code green, GoalScope-steered output purple, pivot token yellow
from typing import Literal
ide_theme: Literal['dark', 'light'] = 'dark'
if ide_theme == 'dark':   # light shades; the task-span fill reuses the text color
    DST_C, PAT_C = 'sea_green3', 'plum2'
    DST_BG       = DST_C
    PIVOT_BG     = 'bright_yellow'
else:                     # the paper's colors (Figure 5)
    DST_C, PAT_C = '#38761d', '#5b277d'
    DST_BG       = '#d9ead3'
    PIVOT_BG     = '#ffe6b2'

repo root: /disk/u/arnab/Codes/Projects/goals


In [2]:
CUDA         = '7'
MODEL        = 'Qwen/Qwen2.5-Coder-32B-Instruct'
GOAL_HEADS   = 'head_sets/Qwen2.5-Coder-32B-Instruct/goal__lam20_thr0.9_band25-56.json'   # the 122 goal heads
DATATYPES_RUN = ['int', 'str', 'list[int]', 'list[str]']
N_PER_DT     = 8            # examples per datatype
USER         = ('Write a Python program that implements a function `func` which, given {phrase} `{arg}`, '
                'computes 3 random but different transformations or computations on the data, stored in the '
                'variables `a`, `b` and `c`.')
ALPHA        = 0.1          # GoalScope strength: the probe's residual h gets + ALPHA * ||h|| along the vector
STEER_LAYERS = (35, 46)     # the probe is steered at layers 35..45
SEED         = 123          # which two computations are prefilled

os.environ['CUDA_VISIBLE_DEVICES'] = CUDA

## Load the model
Qwen2.5-Coder-32B-Instruct and its 122 goal heads.

In [3]:
import re, random, collections
import torch
from rich.console import Console
from rich.table import Table
from rich.text import Text
from rich.markup import escape

from src.models import ModelandTokenizer
from src.tokens import prepare_input
from src.utils import experiment_utils
from src.goals.head_sets import load_json_heads
from src.goals.goalscope import apply_goalscope, _build_causality_prompt, CAUSAL_PREFIX
from src.nrgs.data import QTASKS, DATATYPES, _assign_line, _signature
from scripts.evaluation.eval_goalscope import (
    MAX_NEW_TOKENS, GEN_MAX_NEW_TOKENS, _chat_prompt, generate_completion, frontier_impl_span,
    pivot_positions, hard_eval,
)

console = Console(width=130, highlight=False)
torch.set_grad_enabled(False)
experiment_utils.set_seed(SEED)

mt = ModelandTokenizer(model_key=MODEL, dtype=torch.bfloat16, device_map={'': 0}, attn_implementation='eager')
heads = load_json_heads(GOAL_HEADS, mt.n_layer, mt.n_heads)
gs_kwargs = dict(selected_heads=heads, strategy='additive_norm', alpha=ALPHA, steer_layer_range=STEER_LAYERS,
                 n_steer_tokens='all', max_new_tokens=MAX_NEW_TOKENS)
print(f'goal heads: {len(heads)} (layers {min(l for l, _ in heads)}-{max(l for l, _ in heads)})')

# cut a readout at the first chat-template turn tag
trim = lambda s: re.split(r'<\|(?:user|observation|endoftext|im_end)\|>', s)[0]

Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory: /disk/u/models


goal heads: 122 (layers 28-55)


## 1. The examples and the LM's own `c`
Two computations from the task bank are prefilled per example (no repeated pair within a datatype). The LM writes
the `c` line greedily. The first example's prompt is shown with its four pivots in yellow.

In [4]:
rng = random.Random(SEED)
examples = []
for dt in DATATYPES_RUN:
    bank, arg = QTASKS[dt].predicates, QTASKS[dt].var_names[0]
    seen = set()
    while len([e for e in examples if e['dt'] == dt]) < N_PER_DT:
        pa, pb = rng.sample(bank, 2)
        if (pa.id, pb.id) in seen: continue
        seen.add((pa.id, pb.id))
        prefill = ('<code>\n' + _signature(dt, arg, 'python') + '\n'
                   + _assign_line(pa, arg, 'python', res_name='a') + '\n'
                   + _assign_line(pb, arg, 'python', res_name='b') + '\n    c =')
        prompt = _chat_prompt(mt, USER.format(phrase=DATATYPES[dt]['phrase'], arg=arg), prefill)
        completion = generate_completion(mt, prompt, GEN_MAX_NEW_TOKENS)
        full_text = prompt + completion
        examples.append(dict(dt=dt, a=pa.id, b=pb.id, prefill=prefill, prompt=prompt, full_text=full_text,
                             impl_span=frontier_impl_span(prompt, completion, 'c'),
                             pivs=pivot_positions(mt, prompt, full_text, ['prev_end', 'indent', 'var', 'eq'])))

ex = examples[0]
enc = prepare_input(prompts=ex['prompt'], tokenizer=mt, return_offsets_mapping=True)
start, pos0 = ex['prompt'].index('<|im_start|>user'), set(ex['pivs'].values())
txt = Text()
for i, (tid, (s, e)) in enumerate(zip(enc['input_ids'][0].tolist(), enc['offset_mapping'][0].tolist())):
    if e <= start: continue
    txt.append(mt.tokenizer.decode([tid]), style=f'bold black on {PIVOT_BG}' if i in pos0 else 'dim')
console.rule('example 1: the prompt'); console.print(txt)

t = Table(title=f'{len(examples)} examples: prefilled a, b (task-bank ids) and the c line the LM writes')
t.add_column('datatype'); t.add_column('a'); t.add_column('b'); t.add_column('LM writes', max_width=70)
for e in examples:
    t.add_row(escape(e['dt']), e['a'], e['b'], f"[{DST_C}]{escape(e['impl_span'])}[/]")
console.print(t)

───────────────────────────────────────────────────── example 1: the prompt ──────────────────────────────────────────────────────

<|im_start|>user
Write a Python program that implements a function `func` which, given an integer `n`, computes 3 random but different 
transformations or computations on the data, stored in the variables `a`, `b` and `c`.<|im_end|>
<|im_start|>assistant
<code>
def func(n: int):
    a = (n > 0)
    b = (10 <= abs(n) <= 99)
    c =

                  32 examples: prefilled a, b (task-bank ids) and the c line the LM writes                   
┏━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ datatype  ┃ a                       ┃ b                    ┃ LM writes                                    ┃
┡━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ int       │ is_positive             │ is_two_digit         │ c = (n % 2 == 0)                             │
│ int       │ digit_count             │ absolute_value       │ c = sum(int(digit) for digit in str(abs(n))) │
│ int       │ sign                    │ is_two_digit         │ c = n % 7                                    │
│ int       │ is_multiple_of_3        │ sign                 │ c = n ** 0.5 if n >= 0 else None             │
│ int       │ is_positive             │ is_perfect_square    │ c = sum(int(digit) for digit in str(abs(n))) │
│ int       │ last_digit              │ is_prime             │ c = sum(int(digit) for digit in str(abs(n))) │
│ int       │ is_positive             │ digit_count          │ c = sum(int(digit) for digit in str(abs(n))) │
│ int       │ is_two_digit            │ is_prime             │ c = sum(int(digit) for digit in str(abs(n))) │
│ str       │ all_lowercase           │ has_space            │ c = len(set(s))                              │
│ str       │ count_spaces            │ vowel_count          │ c = ''.join(sorted(set(s)))                  │
│ str       │ is_palindrome           │ last_char            │ c = len(set(s))                              │
│ str       │ first_is_letter         │ is_alnum             │ c = len(set(s))                              │
│ str       │ vowel_count             │ last_char            │ c = len(set(s))                              │
│ str       │ is_alnum                │ length               │ c = s[::-1]                                  │
│ str       │ is_palindrome           │ count_letters        │ c = ''.join(sorted(set(s)))                  │
│ str       │ last_char               │ length               │ c = s[::-1]                                  │
│ list[int] │ total_sum               │ distinct_count       │ c = max(nums) - min(nums)                    │
│ list[int] │ has_duplicate           │ max_val              │ c = sum(nums) / len(nums) if nums else 0     │
│ list[int] │ has_even                │ has_duplicate        │ c = sum(nums) / len(nums) if nums else 0     │
│ list[int] │ all_positive            │ product_all          │ c = sum(x**2 for x in nums)                  │
│ list[int] │ distinct_count          │ has_duplicate        │ c = sum(nums) / len(nums) if nums else 0     │
│ list[int] │ distinct_count          │ all_positive         │ c = sum(nums) / len(nums) if nums else 0     │
│ list[int] │ total_sum               │ is_sorted            │ c = [x**2 for x in nums if x % 2 == 0]       │
│ list[int] │ product_all             │ count_even           │ c = max(nums) - min(nums)                    │
│ list[str] │ total_uppercase_letters │ count_capitalized    │ c = len(set(w.lower() for w in words))       │
│ list[str] │ shortest_word_length    │ has_palindrome       │ c = set(''.join(words))                      │
│ list[str] │ any_empty               │ count_ends_consonant │ c = ''.join(sorted(set(''.join(words))))     │
│ list[str] │ all_unique_lengths      │ shortest_word_length │ c = ''.join(sorted(''.join(words)))          │
│ list[str] │ total_uppercase_letters │ count_short          │ c = ''.join(w[0] for w in words if w)        │
│ list[str] │ all_unique_lengths      │ count_starts_vowel   │ c = ''.join(sorted(''.join(words)))          │
│ list[str] │ total_length            │ all_unique_lengths   │ c = ''.join(sorted(''.join(words)))          │
│ list[str] │ count_starts_vowel      │ total_vowels         │ c = max((len(w) for w in words), default=0)  │
└───────────┴─────────────────────────┴─

## 2. GoalScope at the pivots of every example
First the C++ stub with no vector, to see what it writes on its own. Then, per example and pivot, the description and
the C++ body the goal-head vector produces, each judged against the LM's `c` line (YES = the same computation).

In [5]:
causal_prompt = _build_causality_prompt(mt)
default = trim(generate_completion(mt, causal_prompt, MAX_NEW_TOKENS))
console.rule(f'[bold {DST_C}]C++ stub, no GoalScope[/]')
console.print(f'[grey50]{escape(CAUSAL_PREFIX)}[/][{DST_C}]{escape(default.rstrip())}[/]\n')

PIV = ['prev_end', 'indent', 'var', 'eq']
for e in examples:
    positions = sorted(e['pivs'].values())
    res = apply_goalscope(e['full_text'], mt=mt, token_positions=positions, include_noncode=True, **gs_kwargs)
    faith = {r['pos']: trim(r['patched_gen']) for r in res if r['scoped']}
    res_c = apply_goalscope(e['full_text'], mt=mt, token_positions=positions, goalscope_prompt=causal_prompt, **gs_kwargs)
    causal = {r['pos']: CAUSAL_PREFIX + trim(r['patched_gen']) for r in res_c}
    e['faith'] = {a: faith[p] for a, p in e['pivs'].items()}
    e['causal'] = {a: causal[p] for a, p in e['pivs'].items()}
    e['vf'] = {a: hard_eval(e['faith'][a], e['impl_span'], 'faithfulness')['verdict'] for a in PIV}
    e['vc'] = {a: hard_eval(e['causal'][a], e['impl_span'], 'causality')['verdict'] for a in PIV}

V = lambda v: '[bold green]Y[/]' if v == 'YES' else '[red]n[/]'
one = lambda s, n=70: escape(' '.join(s.split())[:n] + ('…' if len(' '.join(s.split())) > n else ''))
t = Table(title='per example: the LM\'s c, the readouts at `=`, and the verdicts per pivot (faithfulness / causality)',
          show_lines=True)
t.add_column('dt'); t.add_column('LM writes', max_width=34); t.add_column('description at `=`', max_width=40)
t.add_column('C++ stub at `=`', max_width=40)
for a in PIV: t.add_column(a)
for e in examples:
    t.add_row(escape(e['dt']), escape(e['impl_span']), f"[{PAT_C}]{one(e['faith']['eq'])}[/]",
              f"[{PAT_C}]{one(e['causal']['eq'][len(CAUSAL_PREFIX):])}[/]",
              *[f"{V(e['vf'][a])} / {V(e['vc'][a])}" for a in PIV])
console.print(t)

rate = lambda es, key, a: sum(x[key][a] == 'YES' for x in es) / len(es)
t = Table(title='judge YES-rate per pivot: faithfulness / causality (122 goal heads)')
t.add_column('datatype'); t.add_column('n')
for a in PIV + ['mean']: t.add_column(a)
for name, es in [(dt, [e for e in examples if e['dt'] == dt]) for dt in DATATYPES_RUN] + [('all', examples)]:
    f = [rate(es, 'vf', a) for a in PIV]; k = [rate(es, 'vc', a) for a in PIV]
    t.add_row(escape(name), str(len(es)), *[f'{x:.2f} / {y:.2f}' for x, y in zip(f, k)],
              f'{sum(f) / 4:.2f} / {sum(k) / 4:.2f}', style='bold' if name == 'all' else None)
console.print(t)

───────────────────────────────────────────────────── C++ stub, no GoalScope ─────────────────────────────────────────────────────

auto func (auto value){ return value * 2; }

               per example: the LM's c, the readouts at `=`, and the verdicts per pivot (faithfulness / causality)                
┏━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃ dt        ┃ LM writes                ┃ description at `=`       ┃ C++ stub at `=`          ┃ prev_end ┃ indent ┃ var   ┃ eq    ┃
┡━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ int       │ c = (n % 2 == 0)         │ I'm calculating the      │ return (value > 0) ?     │ Y / n    │ Y / n  │ Y / n │ n / n │
│           │                          │ remainder when the       │ value * value : value +  │          │        │       │       │
│           │                          │ number is divided by 7.  │ 5; }                     │          │        │       │       │
├───────────┼──────────────────────────┼──────────────────────────┼──────────────────────────┼──────────┼────────┼───────┼───────┤
│ int       │ c = sum(int(digit) for   │ Calculating the          │ return (value > 0) ?     │ Y / n    │ n / n  │ n / n │ n / n │
│           │ digit in str(abs(n)))    │ factorial of the number  │ (value * value * value)  │          │        │       │       │
│           │                          │ modulo 1000.             │ : (value % 7); }         │          │        │       │       │
├───────────┼──────────────────────────┼──────────────────────────┼──────────────────────────┼──────────┼────────┼───────┼───────┤
│ int       │ c = n % 7                │ Calculating the          │ return (value % 2 == 0)  │ n / n    │ n / n  │ Y / Y │ Y / Y │
│           │                          │ remainder of the number  │ ? value * value :        │          │        │       │       │
│           │                          │ when divided by 7.       │ std::sqrt(value); }      │          │        │       │       │
├───────────┼──────────────────────────┼──────────────────────────┼──────────────────────────┼──────────┼────────┼───────┼───────┤
│ int       │ c = n ** 0.5 if n >= 0   │ Calculating the          │ return (value > 0) ?     │ Y / Y    │ n / n  │ n / n │ n / Y │
│           │ else None                │ factorial of the number. │ value * value : value +  │          │        │       │       │
│           │                          │                          │ 5; }                     │          │        │       │       │
├───────────┼──────────────────────────┼──────────────────────────┼──────────────────────────┼──────────┼────────┼───────┼───────┤
│ int       │ c = sum(int(digit) for   │ Calculating the          │ return (value > 0) ?     │ n / n    │ n / n  │ Y / n │ n / n │
│           │ digit in str(abs(n)))    │ factorial of the number  │ (value * value * value)  │          │        │       │       │
│           │                          │ modulo 7.                │ : (value % 2 == 0 ?      │          │        │       │       │
│           │                          │                          │ value…                   │          │        │       │       │
├───────────┼──────────────────────────┼──────────────────────────┼──────────────────────────┼──────────┼────────┼───────┼───────┤
│ int       │ c = sum(int(digit) for   │ Calculating the          │ return (value > 0) ?     │ n / n    │ n / n  │ n / n │ n / n │
│           │ digit in str(abs(n)))    │ factorial of the number  │ (value * value * value)  │          │        │       │       │
│           │                          │ modulo 17.               │ : (value + (value <<     │          │        │       │       │
│           │                          │                          │ 2));…                    │          │        │       │       │
├───────────┼──────────────────────────┼──────────────────────────┼──────────────────────────┼──────────┼────────┼───────┼───────┤
│ int       │ c = sum(int(digit) for   │ Calculating the          │ re

          judge YES-rate per pivot: faithfulness / causality (122 goal heads)           
┏━━━━━━━━━━━┳━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ datatype  ┃ n  ┃ prev_end    ┃ indent      ┃ var         ┃ eq          ┃ mean        ┃
┡━━━━━━━━━━━╇━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━┩
│ int       │ 8  │ 0.38 / 0.25 │ 0.12 / 0.00 │ 0.38 / 0.12 │ 0.12 / 0.25 │ 0.25 / 0.16 │
│ str       │ 8  │ 0.00 / 0.00 │ 0.50 / 0.00 │ 0.88 / 0.00 │ 0.12 / 0.00 │ 0.38 / 0.00 │
│ list[int] │ 8  │ 0.12 / 0.25 │ 0.38 / 0.75 │ 0.12 / 0.38 │ 0.25 / 0.62 │ 0.22 / 0.50 │
│ list[str] │ 8  │ 0.00 / 0.25 │ 0.00 / 0.12 │ 0.12 / 0.12 │ 0.38 / 0.12 │ 0.12 / 0.16 │
│ all       │ 32 │ 0.12 / 0.19 │ 0.25 / 0.22 │ 0.38 / 0.16 │ 0.22 / 0.25 │ 0.24 / 0.20 │
└───────────┴────┴─────────────┴─────────────┴─────────────┴─────────────┴─────────────┘

## What to expect
Nothing in the prompt says what `c` is, so the readout cannot copy it from a task list as it could in Demo 1. Where
the readout at the pivots names the computation the LM then writes (and the C++ stub implements it), the LM has
settled on its third computation before writing any of it. For reference, on the evaluation corpus, where the next
sub-task *is* listed, the same readout scores 85% (faithfulness) and 87% (causality) averaged over the four pivots
(Demo 1). Change `SEED` to prefill other pairs, or `USER` to word the request differently.